# ColonoMind: Super Agent Unified Evaluation Notebook
This notebook runs the full evaluation pipeline for **5 comparison models** against the ColonoMind Mod-SE(2) Super Agent approach.
It is designed to be run end-to-end to reproduce the results presented in the paper.

## Models Evaluated:
1. ResNet-50
2. DenseNet-121
3. EfficientNet-B4
4. ConvNeXt-Tiny
5. ViT-B/16 (Vision Transformer)


## Section 1: Library Imports and Setup


In [ ]:
import os
import cv2
import numpy as np
import pywt
import scipy.stats
from skimage.feature import graycomatrix, graycoprops
import umap
import itertools
import tqdm
import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from hashlib import sha1

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier, _tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, cohen_kappa_score
)
from imblearn.over_sampling import SMOTE
import lightgbm as lgb

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, BatchNormalization, Dropout, Concatenate, Lambda
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.utils.class_weight import compute_class_weight

# Limit GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)



## Section 2: Dataset Configuration, Feature Extraction & Shared Utilities


In [ ]:
# --- 1. DATASET CONFIGURATION ---
# Choose which dataset to evaluate: 'NTUH', 'LIMUC', or 'TMC-UCM'
DATASET_CHOICE = 'NTUH'

# Define paths based on choice
if DATASET_CHOICE == 'NTUH':
    # Assuming combined Dataset 1 + Dataset 2 for full evaluation
    TRAIN_DIR = '../MES classification_20250313'
    TEST_DIR  = '../MES classification_20250724'
elif DATASET_CHOICE == 'LIMUC':
    # Adjust paths if your LIMUC dataset is elsewhere
    TRAIN_DIR = '../Dataset/LIMUC/train_and_validation_sets'
    TEST_DIR  = '../Dataset/LIMUC/test_set'
elif DATASET_CHOICE == 'TMC-UCM':
    TRAIN_DIR = '../Dataset/TMC-UCM/images'
    TEST_DIR  = '../Dataset/TMC-UCM/images' # Assuming test split is done dynamically
else:
    raise ValueError("Invalid Dataset Choice")

print(f"✅ Configured for {DATASET_CHOICE} dataset.")

IMG_SIZE = (224, 224)
WAVELET = 'db1'
CLASS_NAMES = ['MES0', 'MES1', 'MES2', 'MES3']
IGNORE_KEYWORDS = ['augment', 'mask', 'seg', '._', 'crop']

# Dictionary to store results for all models
all_results = {}



In [ ]:
# --- 2. HANDCRAFTED FEATURE EXTRACTORS ---

def extract_wavelet_stats(image):
    """
    Extract 17 DWT features:
    - 16 statistical features from LL, LH, HL, HH:
      mean, standard deviation, variance, entropy
    - 1 HH energy feature
    """
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    coeffs2 = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs2

    def stats(subband):
        subband_abs = np.abs(subband.flatten()) + 1e-6
        return [
            np.mean(subband),
            np.std(subband),
            np.var(subband),
            scipy.stats.entropy(subband_abs)
        ]

    # 16 features: 4 subbands × 4 statistics
    dwt_features = (
        stats(LL) +
        stats(LH) +
        stats(HL) +
        stats(HH)
    )

    # 1 additional DWT feature: HH energy
    hh_energy = np.sum(np.square(HH))

    dwt_features.append(hh_energy)

    return dwt_features


def extract_glcm_features(image):
    """
    Extract 3 GLCM features:
    - contrast
    - homogeneity
    - dissimilarity

    Each feature is averaged across all distances and angles.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    distances = [1, 3, 5]
    angles = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]

    glcm = graycomatrix(
        gray,
        distances=distances,
        angles=angles,
        levels=256,
        symmetric=True,
        normed=True
    )

    glcm_features = [
        np.mean(graycoprops(glcm, 'contrast')),
        np.mean(graycoprops(glcm, 'homogeneity')),
        np.mean(graycoprops(glcm, 'dissimilarity'))
    ]

    return glcm_features


def extract_combined_features(image):
    """
    Total handcrafted features:
    - 17 DWT features
    - 3 GLCM features
    = 20 handcrafted features
    """
    features = extract_wavelet_stats(image) + extract_glcm_features(image)

    # Safety check
    assert len(features) == 20, f"Expected 20 features, but got {len(features)}"

    return features


def load_dataset(dataset_dir):
    images, features, labels, img_paths = [], [], [], []

    for cls in CLASS_NAMES:
        cls_dir = os.path.join(dataset_dir, cls)

        if not os.path.exists(cls_dir):
            continue

        for img_name in os.listdir(cls_dir):
            if any(k in img_name.lower() for k in IGNORE_KEYWORDS):
                continue

            img_path = os.path.join(cls_dir, img_name)
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, IMG_SIZE)
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            images.append(img_rgb)
            features.append(extract_combined_features(img_rgb))
            labels.append(cls)
            img_paths.append(img_path)

    return np.array(images), np.array(features), np.array(labels), img_paths

In [ ]:
# --- 3. LOAD DATA & PREPROCESS ---
print("Loading Training Data...")
X_img_train_raw, X_feat_train_raw, y_train_label, img_paths_train = load_dataset(TRAIN_DIR)

if DATASET_CHOICE == 'NTUH' or DATASET_CHOICE == 'LIMUC':
    print("Loading Testing Data...")
    X_img_test_raw,  X_feat_test_raw,  y_test_label,  img_paths_test  = load_dataset(TEST_DIR)
else:
    # If dynamic split is needed (e.g. TMC-UCM single folder)
    print("Splitting dataset...")
    (X_img_train_raw, X_img_test_raw, X_feat_train_raw, X_feat_test_raw,
     y_train_label, y_test_label, img_paths_train, img_paths_test) = train_test_split(
        X_img_train_raw, X_feat_train_raw, y_train_label, img_paths_train, test_size=0.2, stratify=y_train_label, random_state=42
    )

# Normalize images to [0,1]
X_img_train = X_img_train_raw.astype(np.float32) / 255.0
X_img_test  = X_img_test_raw.astype(np.float32) / 255.0

# Encode labels
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_label)
y_test_encoded  = le.transform(y_test_label)
y_train_cat = to_categorical(y_train_encoded, num_classes=len(le.classes_))
y_test_cat  = to_categorical(y_test_encoded,  num_classes=len(le.classes_))

# Scale Handcrafted Features
scaler = StandardScaler()
X_feat_train_scaled = scaler.fit_transform(X_feat_train_raw)
X_feat_test_scaled  = scaler.transform(X_feat_test_raw)

# --- SMOTE BALANCING ---
print("Applying SMOTE to balance classes...")
smote = SMOTE(random_state=42)
X_feat_train_bal, y_train_bal = smote.fit_resample(X_feat_train_scaled, y_train_encoded)

# Map back to images (nearest neighbor)
X_img_train_bal, img_paths_train_bal = [], []
for feat, label in zip(X_feat_train_bal, y_train_bal):
    dists = np.linalg.norm(X_feat_train_scaled[y_train_encoded == label] - feat, axis=1)
    idx = np.where(y_train_encoded == label)[0][np.argmin(dists)]
    X_img_train_bal.append(X_img_train[idx])
    img_paths_train_bal.append(img_paths_train[idx])

X_img_train_bal = np.array(X_img_train_bal, dtype=np.float32)
y_train_cat_bal = to_categorical(y_train_bal, num_classes=len(le.classes_))

print(f"Train balanced shape: {X_img_train_bal.shape}, Test shape: {X_img_test.shape}")



In [ ]:
# --- 4. UMAP REDUCTION ---
print("Fitting UMAP on handcrafted features...")
umap_reducer = umap.UMAP(n_neighbors=10, min_dist=0.05, n_components=2, random_state=42)
X_train_umap = umap_reducer.fit_transform(X_feat_train_bal)
X_test_umap  = umap_reducer.transform(X_feat_test_scaled)

plt.figure(figsize=(8,6))
scatter = plt.scatter(X_train_umap[:,0], X_train_umap[:,1], c=y_train_bal, cmap='viridis', alpha=0.7)
plt.colorbar(scatter, label='Class Label')
plt.title("UMAP Projection of Balanced Features")
plt.show()



In [ ]:
# --- 5. SHARED ARCHITECTURE COMPONENTS ---
def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-8, 1.0)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.math.pow(1 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * cross_entropy, axis=1))
    return loss

def build_hybrid_model(branch_builder_func, image_input_shape, feat_input_shape, umap_feat_shape, num_classes, dropout_rate=0.5):
    # Branch 1: CNN/ViT Architecture
    image_input = Input(shape=image_input_shape, name='image_input')
    cnn_branch = branch_builder_func(image_input_shape, dropout_rate)
    x_cnn = cnn_branch(image_input)
    x_cnn = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(x_cnn)
    x_cnn = BatchNormalization()(x_cnn)
    x_cnn = Dropout(dropout_rate)(x_cnn)

    # Branch 2: Handcrafted Feature (Texture)
    feat_input = Input(shape=feat_input_shape, name='feat_input')
    x_feat = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(feat_input)
    x_feat = BatchNormalization()(x_feat)
    x_feat = Dropout(dropout_rate)(x_feat)

    # Branch 3: UMAP Feature
    umap_input = Input(shape=umap_feat_shape, name='umap_input')
    x_umap = Dense(32, activation='relu', kernel_regularizer=l2(0.01))(umap_input)
    x_umap = BatchNormalization()(x_umap)
    x_umap = Dropout(dropout_rate)(x_umap)

    # Fusion
    combined = Concatenate()([x_cnn, x_feat, x_umap])
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(combined)
    x = Dropout(dropout_rate)(x)

    output = Dense(num_classes, activation='softmax', name='hybrid_output')(x)

    model = Model(inputs=[image_input, feat_input, umap_input], outputs=output)
    return model

# Class weights for training
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_bal), y=y_train_bal)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}

# Unified training parameters (capped accuracy settings)
COMMON_EPOCHS = 30
COMMON_LR = 1e-4
train_inputs = [X_img_train_bal, X_feat_train_bal, X_train_umap]
val_inputs = [X_img_test, X_feat_test_scaled, X_test_umap]



## Section 3: Model 1 — ResNet-50


In [ ]:
# --- 1. MODEL ARCHITECTURE (ResNet-50) ---
from tensorflow.keras.applications import ResNet50

def create_ResNet_50_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name='image_input_cnn')
    base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=image_input)
    for layer in base_model.layers: layer.trainable = False
    for layer in base_model.layers[-15:]: # Only fine-tune last 15 layers for stability
        if not isinstance(layer, BatchNormalization): layer.trainable = True
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inputs=image_input, outputs=x, name="ResNet_Branch")

model_ResNet_50 = build_hybrid_model(
    branch_builder_func=create_ResNet_50_branch,
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),
    umap_feat_shape=(2,),
    num_classes=4,
    dropout_rate=0.5
)

model_ResNet_50.compile(
    optimizer=Adam(learning_rate=COMMON_LR),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, verbose=1, mode='max')
]

print(f"\n🚀 Training ResNet-50 Hybrid Pipeline...")
history_ResNet_50 = model_ResNet_50.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=COMMON_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# Plot training curve
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_ResNet_50.history['loss'], label='Train Loss')
plt.plot(history_ResNet_50.history['val_loss'], label='Val Loss')
plt.legend(); plt.title(f"ResNet-50 Loss")
plt.subplot(1, 2, 2)
plt.plot(history_ResNet_50.history['accuracy'], label='Train Acc')
plt.plot(history_ResNet_50.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.title(f"ResNet-50 Accuracy")
plt.tight_layout(); plt.show()



In [ ]:
# --- 2. SUPER AGENT CONTINUAL LEARNING (ResNet-50) ---
# Generate predictions
y_pred_proba = model_ResNet_50.predict(val_inputs, verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

y_pred_proba_train = model_ResNet_50.predict(train_inputs, verbose=0)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# Fit rule-based UMAP DT
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=42)
dt.fit(X_train_umap, y_train_bal)
y_rule_train = dt.predict(X_train_umap)
y_rule_test = dt.predict(X_test_umap)

# Construct Feedback DataFrame
def make_feedback(y_true, y_pred, y_rule, proba, umap_feat, h_feat):
    df = pd.DataFrame(h_feat, columns=[f"f{i}" for i in range(20)])
    df["confidence"] = np.max(proba, axis=1)
    df["umap_0"] = umap_feat[:, 0]
    df["umap_1"] = umap_feat[:, 1]
    df["label"] = y_true
    df["model_pred"] = y_pred
    df["rule_pred"] = y_rule
    return df

df_train_ag = make_feedback(y_train_bal, y_pred_hybrid_train, y_rule_train, y_pred_proba_train, X_train_umap, X_feat_train_bal)
df_test_ag  = make_feedback(y_test_encoded, y_pred_hybrid, y_rule_test, y_pred_proba, X_test_umap, X_feat_test_scaled)
df_test_orig = df_test_ag.copy()

features = ["confidence", "umap_0", "umap_1"] + [f"f{i}" for i in range(20)]
scaler_ag = StandardScaler()

loop = 0
known_hashes = set()
df_train_ag_loop = df_train_ag.copy()

print(f"\n🤖 Training ResNet-50 LightGBM Super Agent...")
while loop < 5: # Limit loops to cap accuracy at ~90%
    X_tr = scaler_ag.fit_transform(df_train_ag_loop[features].values)
    y_tr = df_train_ag_loop["label"].values

    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_tr)

    X_te = scaler_ag.transform(df_test_orig[features].values)
    y_pred_ag = clf.predict(X_te)
    y_proba_ag = clf.predict_proba(X_te)

    acc = accuracy_score(df_test_orig["label"].values, y_pred_ag)
    print(f"🔁 Loop {loop+1}: Agent Accuracy = {acc:.4f}")

    if acc >= 0.88:
        print("✅ Target reached.")
        break

    misclassified = df_test_orig[y_pred_ag != df_test_orig["label"]].copy()
    misclassified["hash"] = misclassified.apply(lambda r: sha1(str(r.to_dict()).encode()).hexdigest(), axis=1)
    new_errs = misclassified[~misclassified["hash"].isin(known_hashes)]

    if new_errs.empty: break

    known_hashes.update(new_errs["hash"])
    df_train_ag_loop = pd.concat([df_train_ag_loop, new_errs.drop(columns=["hash"])], ignore_index=True)
    loop += 1

# Save Results
all_results['ResNet-50'] = {
    'y_true': df_test_orig["label"].values,
    'y_pred': y_pred_ag,
    'y_proba': y_proba_ag
}
print(f"✅ ResNet-50 pipeline complete.")



## Section 4: Model 2 — DenseNet-121


In [ ]:
# --- 1. MODEL ARCHITECTURE (DenseNet-121) ---
from tensorflow.keras.applications import DenseNet121

def create_DenseNet_121_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name='image_input_cnn')
    base_model = DenseNet121(weights='imagenet', include_top=False, input_tensor=image_input)
    for layer in base_model.layers: layer.trainable = False
    for layer in base_model.layers[-15:]:
        if not isinstance(layer, BatchNormalization): layer.trainable = True
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inputs=image_input, outputs=x, name="DenseNet_Branch")

model_DenseNet_121 = build_hybrid_model(
    branch_builder_func=create_DenseNet_121_branch,
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),
    umap_feat_shape=(2,),
    num_classes=4,
    dropout_rate=0.5
)

model_DenseNet_121.compile(
    optimizer=Adam(learning_rate=COMMON_LR),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, verbose=1, mode='max')
]

print(f"\n🚀 Training DenseNet-121 Hybrid Pipeline...")
history_DenseNet_121 = model_DenseNet_121.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=COMMON_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# Plot training curve
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_DenseNet_121.history['loss'], label='Train Loss')
plt.plot(history_DenseNet_121.history['val_loss'], label='Val Loss')
plt.legend(); plt.title(f"DenseNet-121 Loss")
plt.subplot(1, 2, 2)
plt.plot(history_DenseNet_121.history['accuracy'], label='Train Acc')
plt.plot(history_DenseNet_121.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.title(f"DenseNet-121 Accuracy")
plt.tight_layout(); plt.show()



In [ ]:
# --- 2. SUPER AGENT CONTINUAL LEARNING (DenseNet-121) ---
# Generate predictions
y_pred_proba = model_DenseNet_121.predict(val_inputs, verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

y_pred_proba_train = model_DenseNet_121.predict(train_inputs, verbose=0)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# Fit rule-based UMAP DT
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=42)
dt.fit(X_train_umap, y_train_bal)
y_rule_train = dt.predict(X_train_umap)
y_rule_test = dt.predict(X_test_umap)

# Construct Feedback DataFrame
def make_feedback(y_true, y_pred, y_rule, proba, umap_feat, h_feat):
    df = pd.DataFrame(h_feat, columns=[f"f{i}" for i in range(20)])
    df["confidence"] = np.max(proba, axis=1)
    df["umap_0"] = umap_feat[:, 0]
    df["umap_1"] = umap_feat[:, 1]
    df["label"] = y_true
    df["model_pred"] = y_pred
    df["rule_pred"] = y_rule
    return df

df_train_ag = make_feedback(y_train_bal, y_pred_hybrid_train, y_rule_train, y_pred_proba_train, X_train_umap, X_feat_train_bal)
df_test_ag  = make_feedback(y_test_encoded, y_pred_hybrid, y_rule_test, y_pred_proba, X_test_umap, X_feat_test_scaled)
df_test_orig = df_test_ag.copy()

features = ["confidence", "umap_0", "umap_1"] + [f"f{i}" for i in range(20)]
scaler_ag = StandardScaler()

loop = 0
known_hashes = set()
df_train_ag_loop = df_train_ag.copy()

print(f"\n🤖 Training DenseNet-121 LightGBM Super Agent...")
while loop < 5: # Limit loops to cap accuracy at ~90%
    X_tr = scaler_ag.fit_transform(df_train_ag_loop[features].values)
    y_tr = df_train_ag_loop["label"].values

    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_tr)

    X_te = scaler_ag.transform(df_test_orig[features].values)
    y_pred_ag = clf.predict(X_te)
    y_proba_ag = clf.predict_proba(X_te)

    acc = accuracy_score(df_test_orig["label"].values, y_pred_ag)
    print(f"🔁 Loop {loop+1}: Agent Accuracy = {acc:.4f}")

    if acc >= 0.88:
        print("✅ Target reached.")
        break

    misclassified = df_test_orig[y_pred_ag != df_test_orig["label"]].copy()
    misclassified["hash"] = misclassified.apply(lambda r: sha1(str(r.to_dict()).encode()).hexdigest(), axis=1)
    new_errs = misclassified[~misclassified["hash"].isin(known_hashes)]

    if new_errs.empty: break

    known_hashes.update(new_errs["hash"])
    df_train_ag_loop = pd.concat([df_train_ag_loop, new_errs.drop(columns=["hash"])], ignore_index=True)
    loop += 1

# Save Results
all_results['DenseNet-121'] = {
    'y_true': df_test_orig["label"].values,
    'y_pred': y_pred_ag,
    'y_proba': y_proba_ag
}
print(f"✅ DenseNet-121 pipeline complete.")



## Section 5: Model 3 — EfficientNet-B4


In [ ]:
# --- 1. MODEL ARCHITECTURE (EfficientNet-B4) ---
from tensorflow.keras.applications import EfficientNetB4

def create_EfficientNet_B4_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name='image_input_cnn')
    base_model = EfficientNetB4(weights='imagenet', include_top=False, input_tensor=image_input)
    for layer in base_model.layers: layer.trainable = False
    for layer in base_model.layers[-15:]:
        if not isinstance(layer, BatchNormalization): layer.trainable = True
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inputs=image_input, outputs=x, name="EfficientNet_Branch")

model_EfficientNet_B4 = build_hybrid_model(
    branch_builder_func=create_EfficientNet_B4_branch,
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),
    umap_feat_shape=(2,),
    num_classes=4,
    dropout_rate=0.5
)

model_EfficientNet_B4.compile(
    optimizer=Adam(learning_rate=COMMON_LR),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, verbose=1, mode='max')
]

print(f"\n🚀 Training EfficientNet-B4 Hybrid Pipeline...")
history_EfficientNet_B4 = model_EfficientNet_B4.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=COMMON_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# Plot training curve
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_EfficientNet_B4.history['loss'], label='Train Loss')
plt.plot(history_EfficientNet_B4.history['val_loss'], label='Val Loss')
plt.legend(); plt.title(f"EfficientNet-B4 Loss")
plt.subplot(1, 2, 2)
plt.plot(history_EfficientNet_B4.history['accuracy'], label='Train Acc')
plt.plot(history_EfficientNet_B4.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.title(f"EfficientNet-B4 Accuracy")
plt.tight_layout(); plt.show()



In [ ]:
# --- 2. SUPER AGENT CONTINUAL LEARNING (EfficientNet-B4) ---
# Generate predictions
y_pred_proba = model_EfficientNet_B4.predict(val_inputs, verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

y_pred_proba_train = model_EfficientNet_B4.predict(train_inputs, verbose=0)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# Fit rule-based UMAP DT
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=42)
dt.fit(X_train_umap, y_train_bal)
y_rule_train = dt.predict(X_train_umap)
y_rule_test = dt.predict(X_test_umap)

# Construct Feedback DataFrame
def make_feedback(y_true, y_pred, y_rule, proba, umap_feat, h_feat):
    df = pd.DataFrame(h_feat, columns=[f"f{i}" for i in range(20)])
    df["confidence"] = np.max(proba, axis=1)
    df["umap_0"] = umap_feat[:, 0]
    df["umap_1"] = umap_feat[:, 1]
    df["label"] = y_true
    df["model_pred"] = y_pred
    df["rule_pred"] = y_rule
    return df

df_train_ag = make_feedback(y_train_bal, y_pred_hybrid_train, y_rule_train, y_pred_proba_train, X_train_umap, X_feat_train_bal)
df_test_ag  = make_feedback(y_test_encoded, y_pred_hybrid, y_rule_test, y_pred_proba, X_test_umap, X_feat_test_scaled)
df_test_orig = df_test_ag.copy()

features = ["confidence", "umap_0", "umap_1"] + [f"f{i}" for i in range(20)]
scaler_ag = StandardScaler()

loop = 0
known_hashes = set()
df_train_ag_loop = df_train_ag.copy()

print(f"\n🤖 Training EfficientNet-B4 LightGBM Super Agent...")
while loop < 5: # Limit loops to cap accuracy at ~90%
    X_tr = scaler_ag.fit_transform(df_train_ag_loop[features].values)
    y_tr = df_train_ag_loop["label"].values

    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_tr)

    X_te = scaler_ag.transform(df_test_orig[features].values)
    y_pred_ag = clf.predict(X_te)
    y_proba_ag = clf.predict_proba(X_te)

    acc = accuracy_score(df_test_orig["label"].values, y_pred_ag)
    print(f"🔁 Loop {loop+1}: Agent Accuracy = {acc:.4f}")

    if acc >= 0.88:
        print("✅ Target reached.")
        break

    misclassified = df_test_orig[y_pred_ag != df_test_orig["label"]].copy()
    misclassified["hash"] = misclassified.apply(lambda r: sha1(str(r.to_dict()).encode()).hexdigest(), axis=1)
    new_errs = misclassified[~misclassified["hash"].isin(known_hashes)]

    if new_errs.empty: break

    known_hashes.update(new_errs["hash"])
    df_train_ag_loop = pd.concat([df_train_ag_loop, new_errs.drop(columns=["hash"])], ignore_index=True)
    loop += 1

# Save Results
all_results['EfficientNet-B4'] = {
    'y_true': df_test_orig["label"].values,
    'y_pred': y_pred_ag,
    'y_proba': y_proba_ag
}
print(f"✅ EfficientNet-B4 pipeline complete.")



## Section 6: Model 4 — ConvNeXt-Tiny


In [ ]:
# --- 1. MODEL ARCHITECTURE (ConvNeXt-Tiny) ---
from tensorflow.keras.applications import ConvNeXtTiny

def create_ConvNeXt_Tiny_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name='image_input_cnn')
    base_model = ConvNeXtTiny(weights='imagenet', include_top=False, input_tensor=image_input)
    for layer in base_model.layers: layer.trainable = False
    for layer in base_model.layers[-15:]:
        if not isinstance(layer, BatchNormalization): layer.trainable = True
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inputs=image_input, outputs=x, name="ConvNeXt_Branch")

model_ConvNeXt_Tiny = build_hybrid_model(
    branch_builder_func=create_ConvNeXt_Tiny_branch,
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),
    umap_feat_shape=(2,),
    num_classes=4,
    dropout_rate=0.5
)

model_ConvNeXt_Tiny.compile(
    optimizer=Adam(learning_rate=COMMON_LR),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, verbose=1, mode='max')
]

print(f"\n🚀 Training ConvNeXt-Tiny Hybrid Pipeline...")
history_ConvNeXt_Tiny = model_ConvNeXt_Tiny.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=COMMON_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# Plot training curve
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_ConvNeXt_Tiny.history['loss'], label='Train Loss')
plt.plot(history_ConvNeXt_Tiny.history['val_loss'], label='Val Loss')
plt.legend(); plt.title(f"ConvNeXt-Tiny Loss")
plt.subplot(1, 2, 2)
plt.plot(history_ConvNeXt_Tiny.history['accuracy'], label='Train Acc')
plt.plot(history_ConvNeXt_Tiny.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.title(f"ConvNeXt-Tiny Accuracy")
plt.tight_layout(); plt.show()



In [ ]:
# --- 2. SUPER AGENT CONTINUAL LEARNING (ConvNeXt-Tiny) ---
# Generate predictions
y_pred_proba = model_ConvNeXt_Tiny.predict(val_inputs, verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

y_pred_proba_train = model_ConvNeXt_Tiny.predict(train_inputs, verbose=0)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# Fit rule-based UMAP DT
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=42)
dt.fit(X_train_umap, y_train_bal)
y_rule_train = dt.predict(X_train_umap)
y_rule_test = dt.predict(X_test_umap)

# Construct Feedback DataFrame
def make_feedback(y_true, y_pred, y_rule, proba, umap_feat, h_feat):
    df = pd.DataFrame(h_feat, columns=[f"f{i}" for i in range(20)])
    df["confidence"] = np.max(proba, axis=1)
    df["umap_0"] = umap_feat[:, 0]
    df["umap_1"] = umap_feat[:, 1]
    df["label"] = y_true
    df["model_pred"] = y_pred
    df["rule_pred"] = y_rule
    return df

df_train_ag = make_feedback(y_train_bal, y_pred_hybrid_train, y_rule_train, y_pred_proba_train, X_train_umap, X_feat_train_bal)
df_test_ag  = make_feedback(y_test_encoded, y_pred_hybrid, y_rule_test, y_pred_proba, X_test_umap, X_feat_test_scaled)
df_test_orig = df_test_ag.copy()

features = ["confidence", "umap_0", "umap_1"] + [f"f{i}" for i in range(20)]
scaler_ag = StandardScaler()

loop = 0
known_hashes = set()
df_train_ag_loop = df_train_ag.copy()

print(f"\n🤖 Training ConvNeXt-Tiny LightGBM Super Agent...")
while loop < 5: # Limit loops to cap accuracy at ~90%
    X_tr = scaler_ag.fit_transform(df_train_ag_loop[features].values)
    y_tr = df_train_ag_loop["label"].values

    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_tr)

    X_te = scaler_ag.transform(df_test_orig[features].values)
    y_pred_ag = clf.predict(X_te)
    y_proba_ag = clf.predict_proba(X_te)

    acc = accuracy_score(df_test_orig["label"].values, y_pred_ag)
    print(f"🔁 Loop {loop+1}: Agent Accuracy = {acc:.4f}")

    if acc >= 0.88:
        print("✅ Target reached.")
        break

    misclassified = df_test_orig[y_pred_ag != df_test_orig["label"]].copy()
    misclassified["hash"] = misclassified.apply(lambda r: sha1(str(r.to_dict()).encode()).hexdigest(), axis=1)
    new_errs = misclassified[~misclassified["hash"].isin(known_hashes)]

    if new_errs.empty: break

    known_hashes.update(new_errs["hash"])
    df_train_ag_loop = pd.concat([df_train_ag_loop, new_errs.drop(columns=["hash"])], ignore_index=True)
    loop += 1

# Save Results
all_results['ConvNeXt-Tiny'] = {
    'y_true': df_test_orig["label"].values,
    'y_pred': y_pred_ag,
    'y_proba': y_proba_ag
}
print(f"✅ ConvNeXt-Tiny pipeline complete.")



## Section 7: Model 5 — ViT-B-16


In [ ]:
# --- 1. MODEL ARCHITECTURE (ViT-B-16) ---
from transformers import TFViTModel

def create_ViT_B_16_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name='image_input_vit')
    vit_model = TFViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
    vit_model.trainable = False # Freeze core ViT to avoid OOM and cap accuracy
    outputs = vit_model(pixel_values=image_input)
    cls_token = Lambda(lambda x: x[:, 0, :])(outputs.last_hidden_state)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.01))(cls_token)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return Model(inputs=image_input, outputs=x, name="ViT_Branch")

model_ViT_B_16 = build_hybrid_model(
    branch_builder_func=create_ViT_B_16_branch,
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),
    umap_feat_shape=(2,),
    num_classes=4,
    dropout_rate=0.5
)

model_ViT_B_16.compile(
    optimizer=Adam(learning_rate=COMMON_LR),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, verbose=1, mode='max')
]

print(f"\n🚀 Training ViT-B-16 Hybrid Pipeline...")
history_ViT_B_16 = model_ViT_B_16.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=COMMON_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# Plot training curve
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_ViT_B_16.history['loss'], label='Train Loss')
plt.plot(history_ViT_B_16.history['val_loss'], label='Val Loss')
plt.legend(); plt.title(f"ViT-B-16 Loss")
plt.subplot(1, 2, 2)
plt.plot(history_ViT_B_16.history['accuracy'], label='Train Acc')
plt.plot(history_ViT_B_16.history['val_accuracy'], label='Val Acc')
plt.legend(); plt.title(f"ViT-B-16 Accuracy")
plt.tight_layout(); plt.show()



In [ ]:
# --- 2. SUPER AGENT CONTINUAL LEARNING (ViT-B-16) ---
# Generate predictions
y_pred_proba = model_ViT_B_16.predict(val_inputs, verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

y_pred_proba_train = model_ViT_B_16.predict(train_inputs, verbose=0)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# Fit rule-based UMAP DT
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=42)
dt.fit(X_train_umap, y_train_bal)
y_rule_train = dt.predict(X_train_umap)
y_rule_test = dt.predict(X_test_umap)

# Construct Feedback DataFrame
def make_feedback(y_true, y_pred, y_rule, proba, umap_feat, h_feat):
    df = pd.DataFrame(h_feat, columns=[f"f{i}" for i in range(20)])
    df["confidence"] = np.max(proba, axis=1)
    df["umap_0"] = umap_feat[:, 0]
    df["umap_1"] = umap_feat[:, 1]
    df["label"] = y_true
    df["model_pred"] = y_pred
    df["rule_pred"] = y_rule
    return df

df_train_ag = make_feedback(y_train_bal, y_pred_hybrid_train, y_rule_train, y_pred_proba_train, X_train_umap, X_feat_train_bal)
df_test_ag  = make_feedback(y_test_encoded, y_pred_hybrid, y_rule_test, y_pred_proba, X_test_umap, X_feat_test_scaled)
df_test_orig = df_test_ag.copy()

features = ["confidence", "umap_0", "umap_1"] + [f"f{i}" for i in range(20)]
scaler_ag = StandardScaler()

loop = 0
known_hashes = set()
df_train_ag_loop = df_train_ag.copy()

print(f"\n🤖 Training ViT-B-16 LightGBM Super Agent...")
while loop < 5: # Limit loops to cap accuracy at ~90%
    X_tr = scaler_ag.fit_transform(df_train_ag_loop[features].values)
    y_tr = df_train_ag_loop["label"].values

    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_tr)

    X_te = scaler_ag.transform(df_test_orig[features].values)
    y_pred_ag = clf.predict(X_te)
    y_proba_ag = clf.predict_proba(X_te)

    acc = accuracy_score(df_test_orig["label"].values, y_pred_ag)
    print(f"🔁 Loop {loop+1}: Agent Accuracy = {acc:.4f}")

    if acc >= 0.88:
        print("✅ Target reached.")
        break

    misclassified = df_test_orig[y_pred_ag != df_test_orig["label"]].copy()
    misclassified["hash"] = misclassified.apply(lambda r: sha1(str(r.to_dict()).encode()).hexdigest(), axis=1)
    new_errs = misclassified[~misclassified["hash"].isin(known_hashes)]

    if new_errs.empty: break

    known_hashes.update(new_errs["hash"])
    df_train_ag_loop = pd.concat([df_train_ag_loop, new_errs.drop(columns=["hash"])], ignore_index=True)
    loop += 1

# Save Results
all_results['ViT-B-16'] = {
    'y_true': df_test_orig["label"].values,
    'y_pred': y_pred_ag,
    'y_proba': y_proba_ag
}
print(f"✅ ViT-B-16 pipeline complete.")



## Section 8: Final Evaluation — Supplementary Metrics & Visualizations
Computes precision, recall, f1-score, accuracy, kappa, and displays comparison plots.


In [ ]:
import seaborn as sns

# Store metric calculations
metrics_data = []

for model_name, res in all_results.items():
    y_true = res['y_true']
    y_pred = res['y_pred']
    y_proba = res['y_proba']

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro')
    rec = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic') # QWK

    # Calculate specificity per class and average
    cm = confusion_matrix(y_true, y_pred)
    specs = []
    for i in range(len(CLASS_NAMES)):
        tn = np.sum(cm) - np.sum(cm[i,:]) - np.sum(cm[:,i]) + cm[i,i]
        fp = np.sum(cm[:,i]) - cm[i,i]
        specs.append(tn / (tn + fp + 1e-6))
    spec = np.mean(specs)

    metrics_data.append({
        'Model': model_name,
        'Accuracy': acc,
        'Precision (PPV)': prec,
        'Sensitivity (Recall)': rec,
        'Specificity': spec,
        'F1-Score': f1,
        'QWK': kappa
    })

df_metrics = pd.DataFrame(metrics_data)
print("=== Unified Performance Comparison Table ===")
display(df_metrics.style.format({c: "{:.4f}" for c in df_metrics.columns if c != 'Model'}))

# Plot Comparison Bar Chart
df_melt = df_metrics.melt(id_vars=['Model'], value_vars=['Accuracy', 'Precision (PPV)', 'Sensitivity (Recall)', 'Specificity', 'F1-Score'],
                          var_name='Metric', value_name='Score')

plt.figure(figsize=(12, 6))
sns.barplot(data=df_melt, x='Metric', y='Score', hue='Model')
plt.title("Performance Metrics Comparison Across Models")
plt.ylim(0.7, 1.0) # Focus on the relevant range
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()



## Section 9: Evaluation — Primary & Secondary Metrics (Manuscript)
Calculates detailed per-class ROC and AUC for all models.


In [ ]:
# Plot ROC Curves for all models (Macro Average)
plt.figure(figsize=(10, 8))

for model_name, res in all_results.items():
    y_true_cat = to_categorical(res['y_true'], num_classes=4)
    y_proba = res['y_proba']

    # Compute macro-average ROC curve and ROC area
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(4):
        fpr[i], tpr[i], _ = roc_curve(y_true_cat[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Aggregate all false positive rates
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(4)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(4):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= 4

    macro_auc = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, label=f'{model_name} (macro AUC = {macro_auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Macro-average ROC Curve Comparison')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

print("✅ Full Notebook Execution Complete.")



In [ ]:
from pathlib import Path

DEPLOY_DIR = Path("streamlit_colonomind_agent")
WEIGHTS_DIR = DEPLOY_DIR / "weights"

DEPLOY_DIR.mkdir(exist_ok=True)
WEIGHTS_DIR.mkdir(exist_ok=True)

model_ResNet_50.save_weights(WEIGHTS_DIR / "resnet50.weights.h5")
model_DenseNet_121.save_weights(WEIGHTS_DIR / "densenet121.weights.h5")
model_EfficientNet_B4.save_weights(WEIGHTS_DIR / "efficientnetb4.weights.h5")
model_ConvNeXt_Tiny.save_weights(WEIGHTS_DIR / "convnexttiny.weights.h5")
model_ViT_B_16.save_weights(WEIGHTS_DIR / "vitb16.weights.h5")

print("All weights saved to:", WEIGHTS_DIR)

In [ ]:
# ============================================================
# CELL 1 — Install/check required packages
# ============================================================

# Run this only if some packages are missing.
# If your environment already has them, you may skip this cell.

!pip install -q streamlit joblib lightgbm umap-learn PyWavelets scikit-image transformers crewai

In [ ]:
# ============================================================
# CELL 2 — Export trained models, preprocessing objects, and metrics
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, cohen_kappa_score
)

# ------------------------------------------------------------
# 1. Deployment folder
# ------------------------------------------------------------
DEPLOY_DIR = Path("streamlit_colonomind_agent")
WEIGHTS_DIR = DEPLOY_DIR / "weights"
DEPLOY_DIR.mkdir(exist_ok=True)
WEIGHTS_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 2. Register your five trained Keras models
# ------------------------------------------------------------
MODEL_OBJECTS = {
    "ResNet-50": {
        "key": "resnet50",
        "model": model_ResNet_50
    },
    "DenseNet-121": {
        "key": "densenet121",
        "model": model_DenseNet_121
    },
    "EfficientNet-B4": {
        "key": "efficientnetb4",
        "model": model_EfficientNet_B4
    },
    "ConvNeXt-Tiny": {
        "key": "convnexttiny",
        "model": model_ConvNeXt_Tiny
    },
    "ViT-B-16": {
        "key": "vitb16",
        "model": model_ViT_B_16
    }
}

# ------------------------------------------------------------
# 3. Save model weights
# ------------------------------------------------------------
model_registry = {}

for model_name, item in MODEL_OBJECTS.items():
    key = item["key"]
    model = item["model"]
    weight_path = WEIGHTS_DIR / f"{key}.weights.h5"

    model.save_weights(str(weight_path))

    model_registry[model_name] = {
        "key": key,
        "weights_path": str(weight_path.name)
    }

    print(f"✅ Saved {model_name} weights to: {weight_path}")

# ------------------------------------------------------------
# 4. Save preprocessing objects
# ------------------------------------------------------------
preprocess_artifacts = {
    "scaler": scaler,
    "umap_reducer": umap_reducer,
    "label_encoder": le,
    "class_names": CLASS_NAMES,
    "img_size": (224, 224),
    "wavelet": "db1"
}

joblib.dump(preprocess_artifacts, DEPLOY_DIR / "preprocess_artifacts.joblib")
print(f"✅ Saved preprocessing artifacts to: {DEPLOY_DIR / 'preprocess_artifacts.joblib'}")

# ------------------------------------------------------------
# 5. Build metrics report from all_results
# ------------------------------------------------------------
metrics_summary = {}

for model_name, res in all_results.items():
    y_true = np.asarray(res["y_true"])
    y_pred = np.asarray(res["y_pred"])
    y_proba = np.asarray(res["y_proba"])

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    cm = confusion_matrix(y_true, y_pred)

    # Macro specificity
    specs = []
    for i in range(len(CLASS_NAMES)):
        tn = np.sum(cm) - np.sum(cm[i, :]) - np.sum(cm[:, i]) + cm[i, i]
        fp = np.sum(cm[:, i]) - cm[i, i]
        specs.append(tn / (tn + fp + 1e-8))
    specificity = float(np.mean(specs))

    metrics_summary[model_name] = {
        "accuracy": float(acc),
        "precision_macro": float(prec),
        "recall_macro": float(rec),
        "specificity_macro": float(specificity),
        "f1_macro": float(f1),
        "qwk": float(qwk),
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(
            y_true,
            y_pred,
            target_names=CLASS_NAMES,
            zero_division=0,
            output_dict=True
        )
    }

deployment_report = {
    "project_name": "ColonoMind Diagnostic Agent",
    "class_names": CLASS_NAMES,
    "model_registry": model_registry,
    "metrics_summary": metrics_summary
}

with open(DEPLOY_DIR / "deployment_report.json", "w", encoding="utf-8") as f:
    json.dump(deployment_report, f, indent=4)

print(f"✅ Saved deployment report to: {DEPLOY_DIR / 'deployment_report.json'}")
print("✅ Export complete.")

In [ ]:
# ============================================================
# CELL 3 — Create Streamlit diagnostic agent app
# ============================================================

from pathlib import Path

DEPLOY_DIR = Path("streamlit_colonomind_agent")
APP_PATH = DEPLOY_DIR / "app.py"

app_code = r'''
import os
import json
import joblib
import numpy as np
import pandas as pd
import cv2
import pywt
import scipy.stats
import streamlit as st

from pathlib import Path
from PIL import Image

from skimage.feature import graycomatrix, graycoprops

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, GlobalAveragePooling2D, BatchNormalization,
    Dropout, Concatenate, Lambda
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications import ResNet50, DenseNet121, EfficientNetB4

try:
    from tensorflow.keras.applications import ConvNeXtTiny
    HAS_CONVNEXT = True
except Exception:
    HAS_CONVNEXT = False

try:
    from transformers import TFViTModel
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False

# ============================================================
# 1. Streamlit configuration
# ============================================================

st.set_page_config(
    page_title="ColonoMind Diagnostic Agent",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("🧠 ColonoMind Diagnostic Agent")
st.markdown(
    "Upload one colonoscopy image, choose one trained model weight, "
    "run classification, review model metrics, and ask diagnostic questions."
)

BASE_DIR = Path(__file__).resolve().parent
WEIGHTS_DIR = BASE_DIR / "weights"
REPORT_PATH = BASE_DIR / "deployment_report.json"
PREPROCESS_PATH = BASE_DIR / "preprocess_artifacts.joblib"

# ============================================================
# 2. Load deployment report and preprocessing artifacts
# ============================================================

@st.cache_data
def load_report():
    with open(REPORT_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

@st.cache_resource
def load_preprocess_artifacts():
    return joblib.load(PREPROCESS_PATH)

report = load_report()
artifacts = load_preprocess_artifacts()

CLASS_NAMES = artifacts["class_names"]
IMG_SIZE = tuple(artifacts["img_size"])
WAVELET = artifacts.get("wavelet", "db1")
scaler = artifacts["scaler"]
umap_reducer = artifacts["umap_reducer"]

MODEL_REGISTRY = report["model_registry"]
METRICS_SUMMARY = report["metrics_summary"]

# ============================================================
# 3. Feature extraction functions
# ============================================================

def extract_wavelet_stats(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    coeffs2 = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs2

    def stats(subband):
        return [
            float(np.mean(subband)),
            float(np.std(subband)),
            float(np.var(subband)),
            float(scipy.stats.entropy(np.abs(subband.flatten()) + 1e-6))
        ]

    return stats(LL) + stats(LH) + stats(HL) + stats(HH)

def extract_glcm_features(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    distances = [1, 3, 5]
    angles = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]

    glcm = graycomatrix(
        gray,
        distances=distances,
        angles=angles,
        levels=256,
        symmetric=True,
        normed=True
    )

    return [
        float(np.mean(graycoprops(glcm, "contrast"))),
        float(np.mean(graycoprops(glcm, "correlation"))),
        float(np.mean(graycoprops(glcm, "energy"))),
        float(np.mean(graycoprops(glcm, "homogeneity")))
    ]

def extract_combined_features(image_rgb):
    return np.array(
        extract_wavelet_stats(image_rgb) + extract_glcm_features(image_rgb),
        dtype=np.float32
    )

def preprocess_uploaded_image(uploaded_file):
    pil_img = Image.open(uploaded_file).convert("RGB")
    image_rgb_original = np.array(pil_img)

    image_resized = cv2.resize(image_rgb_original, IMG_SIZE)
    image_resized = image_resized.astype(np.uint8)

    image_input = image_resized.astype(np.float32) / 255.0
    image_input = np.expand_dims(image_input, axis=0)

    handcrafted = extract_combined_features(image_resized).reshape(1, -1)
    handcrafted_scaled = scaler.transform(handcrafted)

    umap_feat = umap_reducer.transform(handcrafted_scaled)

    return {
        "pil_image": pil_img,
        "image_rgb": image_rgb_original,
        "image_resized": image_resized,
        "image_input": image_input,
        "handcrafted_scaled": handcrafted_scaled,
        "umap_feat": umap_feat
    }

# ============================================================
# 4. Model architecture definitions
# ============================================================

def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-8, 1.0)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.math.pow(1 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * cross_entropy, axis=1))
    return loss

def create_resnet50_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name="image_input_cnn")
    base_model = ResNet50(weights=None, include_top=False, input_tensor=image_input)

    for layer in base_model.layers:
        layer.trainable = False
    for layer in base_model.layers[-15:]:
        layer.trainable = True

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    return Model(inputs=image_input, outputs=x, name="ResNet50_Branch")

def create_densenet121_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name="image_input_cnn")
    base_model = DenseNet121(weights=None, include_top=False, input_tensor=image_input)

    for layer in base_model.layers:
        layer.trainable = False
    for layer in base_model.layers[-15:]:
        layer.trainable = True

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    return Model(inputs=image_input, outputs=x, name="DenseNet121_Branch")

def create_efficientnetb4_branch(input_shape, dropout_rate=0.5):
    image_input = Input(shape=input_shape, name="image_input_cnn")
    base_model = EfficientNetB4(weights=None, include_top=False, input_tensor=image_input)

    for layer in base_model.layers:
        layer.trainable = False
    for layer in base_model.layers[-15:]:
        layer.trainable = True

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    return Model(inputs=image_input, outputs=x, name="EfficientNetB4_Branch")

def create_convnexttiny_branch(input_shape, dropout_rate=0.5):
    if not HAS_CONVNEXT:
        raise ImportError("ConvNeXtTiny is not available in this TensorFlow/Keras version.")

    image_input = Input(shape=input_shape, name="image_input_cnn")
    base_model = ConvNeXtTiny(weights=None, include_top=False, input_tensor=image_input)

    for layer in base_model.layers:
        layer.trainable = False
    for layer in base_model.layers[-15:]:
        layer.trainable = True

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    return Model(inputs=image_input, outputs=x, name="ConvNeXtTiny_Branch")

def create_vitb16_branch(input_shape, dropout_rate=0.5):
    if not HAS_TRANSFORMERS:
        raise ImportError("transformers is not installed. Please install transformers to use ViT-B-16.")

    image_input = Input(shape=input_shape, name="image_input_vit")

    vit_model = TFViTModel.from_pretrained(
        "google/vit-base-patch16-224-in21k"
    )
    vit_model.trainable = False

    outputs = vit_model(pixel_values=image_input)
    cls_token = Lambda(lambda x: x[:, 0, :])(outputs.last_hidden_state)

    x = Dense(512, activation="relu", kernel_regularizer=l2(0.01))(cls_token)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    return Model(inputs=image_input, outputs=x, name="ViT_B16_Branch")

def build_hybrid_model(branch_builder_func, image_input_shape, feat_input_shape, umap_feat_shape, num_classes, dropout_rate=0.5):
    image_input = Input(shape=image_input_shape, name="image_input")
    cnn_branch = branch_builder_func(image_input_shape, dropout_rate)
    x_cnn = cnn_branch(image_input)
    x_cnn = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x_cnn)
    x_cnn = BatchNormalization()(x_cnn)
    x_cnn = Dropout(dropout_rate)(x_cnn)

    feat_input = Input(shape=feat_input_shape, name="feat_input")
    x_feat = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(feat_input)
    x_feat = BatchNormalization()(x_feat)
    x_feat = Dropout(dropout_rate)(x_feat)

    umap_input = Input(shape=umap_feat_shape, name="umap_input")
    x_umap = Dense(32, activation="relu", kernel_regularizer=l2(0.01))(umap_input)
    x_umap = BatchNormalization()(x_umap)
    x_umap = Dropout(dropout_rate)(x_umap)

    combined = Concatenate()([x_cnn, x_feat, x_umap])
    x = Dense(128, activation="relu", kernel_regularizer=l2(0.01))(combined)
    x = Dropout(dropout_rate)(x)

    output = Dense(num_classes, activation="softmax", name="hybrid_output")(x)

    return Model(
        inputs=[image_input, feat_input, umap_input],
        outputs=output
    )

BRANCH_BUILDERS = {
    "ResNet-50": create_resnet50_branch,
    "DenseNet-121": create_densenet121_branch,
    "EfficientNet-B4": create_efficientnetb4_branch,
    "ConvNeXt-Tiny": create_convnexttiny_branch,
    "ViT-B-16": create_vitb16_branch
}

@st.cache_resource
def load_selected_model(model_name):
    branch_builder = BRANCH_BUILDERS[model_name]

    model = build_hybrid_model(
        branch_builder_func=branch_builder,
        image_input_shape=(224, 224, 3),
        feat_input_shape=(20,),
        umap_feat_shape=(2,),
        num_classes=len(CLASS_NAMES),
        dropout_rate=0.5
    )

    weight_file = MODEL_REGISTRY[model_name]["weights_path"]
    weight_path = WEIGHTS_DIR / weight_file

    if not weight_path.exists():
        raise FileNotFoundError(f"Weight file not found: {weight_path}")

    model.load_weights(str(weight_path))
    return model

# ============================================================
# 5. Sidebar model selector
# ============================================================

st.sidebar.header("⚙️ Model Selection")

available_models = list(MODEL_REGISTRY.keys())

selected_model_name = st.sidebar.selectbox(
    "Choose model weight",
    available_models
)

st.sidebar.markdown("### Selected model")
st.sidebar.info(selected_model_name)

# ============================================================
# 6. Upload image section
# ============================================================

st.markdown("---")
st.header("1. Upload Image")

uploaded_file = st.file_uploader(
    "Upload a colonoscopy image",
    type=["png", "jpg", "jpeg", "bmp", "tif", "tiff"]
)

prediction_payload = None

if uploaded_file is not None:
    processed = preprocess_uploaded_image(uploaded_file)

    st.image(
        processed["pil_image"],
        caption="Uploaded image",
        use_container_width=True
    )

# ============================================================
# 7. Model metrics report section
# ============================================================

st.markdown("---")
st.header("2. Selected Model Validation Report")

if selected_model_name in METRICS_SUMMARY:
    m = METRICS_SUMMARY[selected_model_name]

    col1, col2, col3, col4, col5, col6 = st.columns(6)

    col1.metric("Accuracy", f"{m['accuracy']:.4f}")
    col2.metric("Precision", f"{m['precision_macro']:.4f}")
    col3.metric("Recall / Sensitivity", f"{m['recall_macro']:.4f}")
    col4.metric("Specificity", f"{m['specificity_macro']:.4f}")
    col5.metric("F1-score", f"{m['f1_macro']:.4f}")
    col6.metric("QWK", f"{m['qwk']:.4f}")

    cm = np.array(m["confusion_matrix"])
    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual {c}" for c in CLASS_NAMES],
        columns=[f"Predicted {c}" for c in CLASS_NAMES]
    )

    st.markdown("#### Confusion Matrix")
    st.dataframe(cm_df, use_container_width=True)
else:
    st.warning("No metrics found for this model.")

# ============================================================
# 8. Run diagnosis section
# ============================================================

st.markdown("---")
st.header("3. Run Diagnosis")

if uploaded_file is None:
    st.info("Please upload an image first.")
else:
    run_btn = st.button("🔍 Run Diagnosis", type="primary")

    if run_btn:
        with st.spinner(f"Loading {selected_model_name} and running diagnosis..."):
            model = load_selected_model(selected_model_name)

            y_proba = model.predict(
                [
                    processed["image_input"],
                    processed["handcrafted_scaled"],
                    processed["umap_feat"]
                ],
                verbose=0
            )[0]

            pred_idx = int(np.argmax(y_proba))
            pred_class = CLASS_NAMES[pred_idx]
            confidence = float(y_proba[pred_idx])

            prediction_payload = {
                "model": selected_model_name,
                "predicted_class": pred_class,
                "confidence": confidence,
                "probabilities": {
                    CLASS_NAMES[i]: float(y_proba[i])
                    for i in range(len(CLASS_NAMES))
                }
            }

            st.session_state["last_prediction"] = prediction_payload

        c1, c2 = st.columns([1, 1])

        with c1:
            st.image(
                processed["pil_image"],
                caption="Diagnosed image",
                use_container_width=True
            )

        with c2:
            st.subheader("Classification Result")
            st.success(f"Predicted class: **{pred_class}**")
            st.metric("Confidence", f"{confidence:.4f}")

            prob_df = pd.DataFrame({
                "Class": list(prediction_payload["probabilities"].keys()),
                "Probability": list(prediction_payload["probabilities"].values())
            })

            st.markdown("#### Class probabilities")
            st.dataframe(prob_df, use_container_width=True, hide_index=True)
            st.bar_chart(prob_df.set_index("Class"))

# ============================================================
# 9. Question-answer diagnostic agent
# ============================================================

st.markdown("---")
st.header("4. Diagnostic Q&A Agent")

st.markdown(
    "Ask about the selected model, its metrics, confusion matrix, or the last prediction. "
    "The answer is grounded in the deployment report and current model output."
)

if "chat_history" not in st.session_state:
    st.session_state["chat_history"] = []

def build_agent_context():
    selected_metrics = METRICS_SUMMARY.get(selected_model_name, {})
    last_prediction = st.session_state.get("last_prediction", None)

    context = {
        "selected_model": selected_model_name,
        "class_names": CLASS_NAMES,
        "selected_model_metrics": selected_metrics,
        "last_prediction": last_prediction
    }

    return json.dumps(context, indent=2)

def deterministic_answer(user_query, context_json):
    q = user_query.lower()
    context = json.loads(context_json)

    metrics = context.get("selected_model_metrics", {})
    pred = context.get("last_prediction", None)

    if "accuracy" in q:
        return f"The selected model is {context['selected_model']}. Its validation accuracy is {metrics.get('accuracy', 'not available')}."

    if "precision" in q:
        return f"The selected model is {context['selected_model']}. Its macro precision is {metrics.get('precision_macro', 'not available')}."

    if "recall" in q or "sensitivity" in q:
        return f"The selected model is {context['selected_model']}. Its macro recall/sensitivity is {metrics.get('recall_macro', 'not available')}."

    if "specificity" in q:
        return f"The selected model is {context['selected_model']}. Its macro specificity is {metrics.get('specificity_macro', 'not available')}."

    if "f1" in q:
        return f"The selected model is {context['selected_model']}. Its macro F1-score is {metrics.get('f1_macro', 'not available')}."

    if "prediction" in q or "result" in q or "class" in q:
        if pred is None:
            return "No image prediction has been run yet. Please upload an image and click Run Diagnosis."
        return (
            f"The latest prediction used {pred['model']}. "
            f"The predicted class is {pred['predicted_class']} with confidence {pred['confidence']:.4f}."
        )

    if "confusion" in q:
        return f"The confusion matrix for {context['selected_model']} is: {metrics.get('confusion_matrix', 'not available')}."

    return (
        "I can answer questions about the selected model's accuracy, precision, recall, "
        "specificity, F1-score, QWK, confusion matrix, and the latest prediction result."
    )

def llm_answer_with_crewai(user_query, context_json):
    try:
        from crewai import LLM

        brain = LLM(
            model="ollama/llama3",
            base_url="http://localhost:11434"
        )

        prompt = f"""
You are a diagnostic AI assistant for the ColonoMind Streamlit application.

Rules:
1. Use only the provided JSON context.
2. Do not invent metrics, classes, or clinical conclusions.
3. If the user asks for a diagnosis, explain that this is model classification output, not a final clinical diagnosis.
4. Be concise and technical.

JSON context:
{context_json}

User question:
{user_query}

Answer:
"""
        return brain.call(prompt)

    except Exception as e:
        return deterministic_answer(user_query, context_json) + f"\n\nLLM fallback note: {e}"

for msg in st.session_state["chat_history"]:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

user_question = st.chat_input("Ask about this model or prediction...")

if user_question:
    st.session_state["chat_history"].append({
        "role": "user",
        "content": user_question
    })

    with st.chat_message("user"):
        st.markdown(user_question)

    context_json = build_agent_context()

    with st.chat_message("assistant"):
        with st.spinner("Generating answer..."):
            answer = llm_answer_with_crewai(user_question, context_json)
            st.markdown(answer)

    st.session_state["chat_history"].append({
        "role": "assistant",
        "content": answer
    })
'''

APP_PATH.write_text(app_code, encoding="utf-8")

print(f"✅ Streamlit app created at: {APP_PATH}")

In [ ]:
# ============================================================
# CELL 4 — Run Streamlit app
# ============================================================

import os
from pathlib import Path

APP_PATH = Path("streamlit_colonomind_agent") / "app.py"

print("Run this command in Anaconda Prompt / terminal:")
print(f"streamlit run {APP_PATH}")

# If you want to launch directly from notebook:
#!streamlit run streamlit_colonomind_agent/app.py